# Session 1 — GPU Benchmark: Batch Size Scaling

## Background

The GPU executes eagerly on thousands of SIMT cores; each step dispatches work immediately. The TPU's systolic array (MXU) processes large matrix blocks in a fixed pipeline — it is most efficient when there is enough work to fill every cell. Batch size is the simplest lever that controls how much work arrives per step and therefore which execution model wins. Session 1 asks: where do these two devices diverge, and by how much?

## Goal

At the end of this session, participants will have run the same training loop on CPU, GPU, and TPU without changing model code. They will be able to read a throughput curve and explain why the GPU saturates early while the TPU scales near-linearly — grounding the interpretation in memory bandwidth and MXU utilization, not hardware marketing.

## Question

At what batch size does the TPU overtake the GPU, and how large does the gap become?

---

**Runtime:** GPU — `Runtime → Change runtime type → T4 GPU`

After running, results are saved to `results/gpu.json` and loaded automatically by `04-analysis.ipynb`.

In [ ]:
import time, torch, torch.nn as nn

assert torch.cuda.is_available(), "No GPU found. Runtime → Change runtime type → T4 GPU."

print(f"Device  : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
D_MODEL, N_HEAD, DIM_FF, SEQ_LEN = 512, 8, 2048, 128
N_STEPS, WARMUP = 50, 5

# ── Model ─────────────────────────────────────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn  = nn.MultiheadAttention(D_MODEL, N_HEAD, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(D_MODEL, DIM_FF), nn.GELU(), nn.Linear(DIM_FF, D_MODEL))
        self.norm1 = nn.LayerNorm(D_MODEL)
        self.norm2 = nn.LayerNorm(D_MODEL)
    def forward(self, x):
        x = self.norm1(x + self.attn(x, x, x)[0])
        x = self.norm2(x + self.ff(x))
        return x

# ── Benchmark function ────────────────────────────────────────────────────────
def benchmark(batch_size, device):
    model     = TransformerBlock().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()
    elapsed = 0.0
    for step in range(N_STEPS + WARMUP):
        # Fresh data every step — avoids caching the same tensor across iterations
        x = torch.randn(batch_size, SEQ_LEN, D_MODEL, device=device)
        y = torch.randn(batch_size, SEQ_LEN, D_MODEL, device=device)
        # Sync before starting the timer so previous GPU work doesn't bleed in
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        if step >= WARMUP:
            elapsed += t1 - t0
    return (N_STEPS * batch_size) / elapsed

print("Model and benchmark function defined.")

In [ ]:
BATCH_SIZES = [4, 8, 16, 32, 64, 128, 256, 512, 1024]
device      = torch.device("cuda")
results     = {}

for bs in BATCH_SIZES:
    try:
        print(f"  [GPU] batch={bs:>4} ... ", end="", flush=True)
        tput = benchmark(bs, device)
        results[bs] = round(tput, 1)
        print(f"{tput:,.0f} samples/sec")
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("OOM — stopping.")
            torch.cuda.empty_cache()
            break
        raise

print("\nDone.")
print("\n── Copy this into 04-analysis.ipynb ──────────────────")
print(f"GPU_RESULTS = {results}")

In [ ]:
import json, os
from datetime import datetime, timezone

def save_results(hardware, results, device_name="", results_dir="results"):
    os.makedirs(results_dir, exist_ok=True)
    payload = {
        "hardware":    hardware,
        "device_name": device_name,
        "session":     "1",
        "timestamp":   datetime.now(timezone.utc).isoformat(),
        "results":     {str(k): v for k, v in sorted(results.items())},
    }
    path = os.path.join(results_dir, f"{hardware.lower()}.json")
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    return path

path = save_results("GPU", results, device_name=torch.cuda.get_device_name(0), results_dir="results")
print(f"Saved → {path}")